# Small-Molecule Dihedral Analysis

這份 Notebook 一次分析 1 個 PSF + 1 個 DCD，適用於不同種類的小分子，不限定 TO。

流程：

1. 設定 PSF 與 DCD 路徑。
2. 讓程式列出可能的小分子 residue。
3. 設定小分子的 resname，以及 φ₁、φ₂ 各四顆原子的名稱。
4. 執行計算，輸出 NPZ、CSV 與圖片。

程式可以協助找出候選小分子及列出 atom names，但無法自動判斷哪四顆原子構成具有研究意義的二面角；二面角仍需依分子結構、參數檔與研究目的指定。


## 1. 環境準備

需要 Python 3、MDAnalysis、NumPy 與 Matplotlib。

    pip install MDAnalysis numpy matplotlib


In [ ]:
# ============================================================
# 2. 載入套件
# ============================================================

from pathlib import Path
import csv

import MDAnalysis as mda
from MDAnalysis.lib.distances import calc_dihedrals
import matplotlib.pyplot as plt
import numpy as np

print(f"MDAnalysis: {mda.__version__}")
print(f"NumPy: {np.__version__}")


## 3. 路徑與分析設定

先修改 PSF_PATH、DCD_PATH、SYSTEM_NAME。

研究時間設定：

- 每個原始 DCD frame = 0.002 ns
- STRIDE = 10 時，每 0.02 ns 取樣一次


In [ ]:
# ============================================================
# 4. 路徑與分析設定：先修改這一區
# ============================================================

PSF_PATH = Path("/path/to/step3_input.psf")
DCD_PATH = Path("/path/to/trajectory.dcd")

SYSTEM_NAME = "example_ligand_rep1"

TIME_PER_FRAME_NS = 0.002
STRIDE = 10

PDF_BINS = 36
ANGLE_RANGE_DEG = (-90.0, 90.0)

OUTPUT_DIR = Path("./results") / SYSTEM_NAME

print(f"System: {SYSTEM_NAME}")
print(f"Sampling interval: {TIME_PER_FRAME_NS * STRIDE:.3f} ns")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 5. 尋找可能的小分子

此步驟只讀取 PSF，不需要先載入 DCD。

程式會排除：

- MDAnalysis 能辨識的 protein、nucleic 與 water
- 只有一顆原子的 residue（通常為離子）
- EXCLUDED_RESNAMES 中列出的常見溶劑與離子

若系統包含 lipid、cofactor 或其他非蛋白質分子，它們也可能出現在候選清單；請依實際系統判斷。


In [ ]:
# ============================================================
# 6. 列出候選小分子
# ============================================================

EXCLUDED_RESNAMES = {
    "TIP3", "TIP3P", "SOL", "HOH", "WAT",
    "POT", "SOD", "CLA", "K", "NA", "CL",
}

if not PSF_PATH.is_file():
    raise FileNotFoundError(
        f"找不到 PSF：{PSF_PATH}\n"
        "請先修改路徑設定區中的 PSF_PATH。"
    )

topology_universe = mda.Universe(str(PSF_PATH))

# 以 MDAnalysis 內建分類排除蛋白質與核酸；水由 EXCLUDED_RESNAMES 排除。
excluded_resindices = set(
    topology_universe.select_atoms(
        "protein or nucleic"
    ).resindices
)

candidate_residues = []

for residue in topology_universe.residues:
    if residue.resindex in excluded_resindices:
        continue
    if residue.resname.upper() in EXCLUDED_RESNAMES:
        continue
    if len(residue.atoms) <= 1:
        continue

    candidate_residues.append(residue)

if not candidate_residues:
    print("沒有找到候選小分子。請檢查 EXCLUDED_RESNAMES 與 topology。")
else:
    print("Possible ligand residues:")
    print("-" * 72)

    for residue in candidate_residues:
        print(
            f"resname={residue.resname:<8} "
            f"resid={residue.resid:<6} "
            f"segid={residue.segid:<8} "
            f"atoms={len(residue.atoms)}"
        )


## 7. 指定小分子與二面角

根據上方候選清單修改：

- LIGAND_RESNAME：小分子的 residue name
- LIGAND_RESID：若同一 resname 只出現一次，可保持 None；若出現多次，填入要分析的 resid
- LIGAND_SEGID：通常保持 None；需要進一步區分時再填
- PHI1_ATOMS、PHI2_ATOMS：依序填入構成二面角的四顆 atom names

TO 範例：

    LIGAND_RESNAME = "TOG"
    PHI1_ATOMS = ["S", "C4", "C6", "C2"]
    PHI2_ATOMS = ["C2", "C6", "C4", "N1"]


In [ ]:
# ============================================================
# 8. 小分子與二面角設定：確認候選 residue 後修改
# ============================================================

LIGAND_RESNAME = "LIG"

# 若同一 resname 有多個 residue，請填入指定值，例如 25。
LIGAND_RESID = None
LIGAND_SEGID = None

PHI1_ATOMS = ["A1", "A2", "A3", "A4"]
PHI2_ATOMS = ["B1", "B2", "B3", "B4"]

PHI1_LABEL = r"$\phi_1$"
PHI2_LABEL = r"$\phi_2$"


def build_ligand_selection():
    """依 resname、選填的 resid 與 segid 建立 selection。"""
    parts = [f"resname {LIGAND_RESNAME}"]

    if LIGAND_RESID is not None:
        parts.append(f"resid {LIGAND_RESID}")

    if LIGAND_SEGID is not None:
        parts.append(f"segid {LIGAND_SEGID}")

    return " and ".join(parts)


LIGAND_SELECTION = build_ligand_selection()
print(f"Ligand selection: {LIGAND_SELECTION}")


## 9. 檢查小分子與 atom names

這一步會顯示選定小分子的所有 atom names。請確認 PHI1_ATOMS 與 PHI2_ATOMS 中的名稱都存在，且每個名稱只對應一顆原子。


In [ ]:
# ============================================================
# 10. 顯示選定小分子的原子
# ============================================================

ligand_preview = topology_universe.select_atoms(LIGAND_SELECTION)

if len(ligand_preview) == 0:
    raise ValueError(
        f"沒有選到小分子：{LIGAND_SELECTION}\n"
        "請回到小分子設定區確認 resname、resid 與 segid。"
    )

selected_residues = ligand_preview.residues

if len(selected_residues) != 1:
    residue_info = [
        f"{res.resname}:{res.resid}:{res.segid}"
        for res in selected_residues
    ]
    raise ValueError(
        "目前 selection 選到多個 residue："
        f"{residue_info}\n"
        "請設定 LIGAND_RESID，必要時再設定 LIGAND_SEGID。"
    )

print(
    f"Selected ligand: resname={selected_residues[0].resname}, "
    f"resid={selected_residues[0].resid}, "
    f"segid={selected_residues[0].segid}"
)
print(f"Number of atoms: {len(ligand_preview)}")
print("Atom names:")
print(" ".join(ligand_preview.names))


## 11. 二面角計算函數

角度會折疊至 −90° 到 90°，延續本研究原本的 torsion 表示方式。若其他研究需要完整 −180° 到 180°，請將 FOLD_TO_90 設為 False。


In [ ]:
# ============================================================
# 12. 計算函數
# ============================================================

FOLD_TO_90 = True


def validate_dihedral_definition(atom_names, label):
    """確認二面角定義包含四個 atom names。"""
    if len(atom_names) != 4:
        raise ValueError(
            f"{label} 必須提供四個 atom names，目前為：{atom_names}"
        )


def fold_angle(angle_deg):
    """將角度折疊至 [-90, 90)。"""
    angle_deg = np.asarray(angle_deg, dtype=float)
    return (angle_deg + 90.0) % 180.0 - 90.0


def select_dihedral_atoms(universe, atom_names, label):
    """
    依設定順序選取二面角的四顆原子。

    每個 atom name 必須在選定的小分子 residue 中恰好出現一次。
    """
    selected_atoms = []

    for atom_name in atom_names:
        selection = (
            f"({LIGAND_SELECTION}) and name {atom_name}"
        )
        atom_group = universe.select_atoms(selection)

        if len(atom_group) != 1:
            raise ValueError(
                f"{label} 的 atom name '{atom_name}' 應選到 1 顆原子，"
                f"實際選到 {len(atom_group)} 顆。\n"
                f"Selection: {selection}\n"
                "請核對上方列出的 atom names。"
            )

        selected_atoms.append(atom_group)

    return selected_atoms


def calculate_single_dihedral(atom_groups, box):
    """計算目前 frame 中四顆原子的二面角，輸出 degree。"""
    angle_rad = calc_dihedrals(
        atom_groups[0].positions[0],
        atom_groups[1].positions[0],
        atom_groups[2].positions[0],
        atom_groups[3].positions[0],
        box=box,
    )

    return float(np.rad2deg(angle_rad))


def calculate_trajectory(psf_path, dcd_path):
    """
    計算單一 DCD 中的小分子 φ₁ 與 φ₂。

    Returns
    -------
    time_ns : numpy.ndarray
        取樣時間，單位為 ns。
    phi1_deg, phi2_deg : numpy.ndarray
        二面角，單位為 degree。
    total_frames : int
        DCD 原始 frame 數。
    """
    universe = mda.Universe(str(psf_path), str(dcd_path))

    phi1_atom_groups = select_dihedral_atoms(
        universe,
        PHI1_ATOMS,
        "PHI1_ATOMS",
    )
    phi2_atom_groups = select_dihedral_atoms(
        universe,
        PHI2_ATOMS,
        "PHI2_ATOMS",
    )

    phi1_values = []
    phi2_values = []

    for ts in universe.trajectory[::STRIDE]:
        box = (
            ts.dimensions
            if ts.dimensions is not None and np.all(ts.dimensions[:3] > 0)
            else None
        )

        phi1_values.append(
            calculate_single_dihedral(phi1_atom_groups, box)
        )
        phi2_values.append(
            calculate_single_dihedral(phi2_atom_groups, box)
        )

    if not phi1_values:
        raise RuntimeError(
            "DCD 沒有產生任何取樣 frame，請檢查軌跡與 STRIDE。"
        )

    phi1_deg = np.asarray(phi1_values)
    phi2_deg = np.asarray(phi2_values)

    if FOLD_TO_90:
        phi1_deg = fold_angle(phi1_deg)
        phi2_deg = fold_angle(phi2_deg)

    time_ns = np.arange(len(phi1_deg)) * TIME_PER_FRAME_NS * STRIDE

    return time_ns, phi1_deg, phi2_deg, len(universe.trajectory)


validate_dihedral_definition(PHI1_ATOMS, "PHI1_ATOMS")
validate_dihedral_definition(PHI2_ATOMS, "PHI2_ATOMS")


## 13. 執行計算並保存

輸出：

- ligand_dihedral_data.npz
- ligand_dihedral_data.csv
- run_summary.txt


In [ ]:
# ============================================================
# 14. 執行計算與儲存
# ============================================================

if not PSF_PATH.is_file():
    raise FileNotFoundError(f"找不到 PSF：{PSF_PATH}")

if not DCD_PATH.is_file():
    raise FileNotFoundError(f"找不到 DCD：{DCD_PATH}")

if not isinstance(STRIDE, int) or STRIDE <= 0:
    raise ValueError("STRIDE 必須是正整數。")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading: {DCD_PATH}")
time_ns, phi1_deg, phi2_deg, total_frames = calculate_trajectory(
    PSF_PATH,
    DCD_PATH,
)

npz_path = OUTPUT_DIR / "ligand_dihedral_data.npz"
np.savez_compressed(
    npz_path,
    system_name=np.asarray(SYSTEM_NAME),
    ligand_selection=np.asarray(LIGAND_SELECTION),
    phi1_atoms=np.asarray(PHI1_ATOMS),
    phi2_atoms=np.asarray(PHI2_ATOMS),
    time_ns=time_ns,
    phi1_deg=phi1_deg,
    phi2_deg=phi2_deg,
    time_per_frame_ns=np.asarray(TIME_PER_FRAME_NS),
    stride=np.asarray(STRIDE),
    fold_to_90=np.asarray(FOLD_TO_90),
)

csv_path = OUTPUT_DIR / "ligand_dihedral_data.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["sampled_frame", "time_ns", "phi1_deg", "phi2_deg"])

    for frame_index, (time_value, phi1, phi2) in enumerate(
        zip(time_ns, phi1_deg, phi2_deg)
    ):
        writer.writerow([
            frame_index,
            f"{time_value:.6f}",
            f"{phi1:.6f}",
            f"{phi2:.6f}",
        ])

summary_path = OUTPUT_DIR / "run_summary.txt"
with summary_path.open("w", encoding="utf-8") as summary_file:
    summary_file.write(f"System: {SYSTEM_NAME}\n")
    summary_file.write(f"PSF: {PSF_PATH}\n")
    summary_file.write(f"DCD: {DCD_PATH}\n")
    summary_file.write(f"Ligand selection: {LIGAND_SELECTION}\n")
    summary_file.write(f"PHI1 atoms: {' - '.join(PHI1_ATOMS)}\n")
    summary_file.write(f"PHI2 atoms: {' - '.join(PHI2_ATOMS)}\n")
    summary_file.write(f"Fold to [-90, 90): {FOLD_TO_90}\n")
    summary_file.write(f"Original frames: {total_frames}\n")
    summary_file.write(f"Sampled frames: {len(time_ns)}\n")
    summary_file.write(f"Time per original frame: {TIME_PER_FRAME_NS} ns\n")
    summary_file.write(f"Stride: {STRIDE}\n")

print(f"Original frames: {total_frames:,}")
print(f"Sampled frames: {len(time_ns):,}")
print(f"Final sampled time: {time_ns[-1]:.3f} ns")
print(f"NPZ saved: {npz_path.resolve()}")
print(f"CSV saved: {csv_path.resolve()}")
print(f"Summary saved: {summary_path.resolve()}")


## 15. 從 NPZ 讀取並繪圖

若只需修改圖片格式，可從此處往下執行，不必重新讀取 DCD。


In [ ]:
# ============================================================
# 16. 載入 NPZ
# ============================================================

npz_path = OUTPUT_DIR / "ligand_dihedral_data.npz"

if not npz_path.is_file():
    raise FileNotFoundError(
        f"找不到 NPZ：{npz_path}\n"
        "請先執行計算 cell。"
    )

data = np.load(npz_path, allow_pickle=False)

required_keys = {
    "system_name",
    "time_ns",
    "phi1_deg",
    "phi2_deg",
}

missing_keys = required_keys.difference(data.files)

if missing_keys:
    raise KeyError(f"NPZ 缺少欄位：{sorted(missing_keys)}")

plot_system_name = str(data["system_name"])
plot_time_ns = data["time_ns"]
plot_phi1 = data["phi1_deg"]
plot_phi2 = data["phi2_deg"]

plot_angle_range = (
    (-90.0, 90.0)
    if bool(data["fold_to_90"])
    else (-180.0, 180.0)
)

print(f"Loaded: {npz_path.resolve()}")


In [ ]:
# ============================================================
# 17. φ₁、φ₂ 時間序列圖
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(24, 9), dpi=300)

plot_items = [
    (plot_phi1, PHI1_LABEL),
    (plot_phi2, PHI2_LABEL),
]

for ax, (angles, angle_label) in zip(axes, plot_items):
    ax.plot(
        plot_time_ns,
        angles,
        color="royalblue",
        linewidth=1.5,
        alpha=0.85,
    )
    ax.set_xlabel("Time (ns)", fontsize=30)
    ax.set_ylabel(f"{angle_label} Angle (degree)", fontsize=30)
    ax.set_title(f"{angle_label} Time Series", fontsize=30)
    ax.set_ylim(*plot_angle_range)
    ax.tick_params(axis="both", which="major", labelsize=24)

fig.tight_layout()

time_figure_path = OUTPUT_DIR / "ligand_dihedral_time_series.png"
fig.savefig(time_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved: {time_figure_path.resolve()}")


In [ ]:
# ============================================================
# 18. φ₁、φ₂ Probability Density
# ============================================================

density_results = []

for angles in (plot_phi1, plot_phi2):
    density, bin_edges = np.histogram(
        angles,
        bins=PDF_BINS,
        range=plot_angle_range,
        density=True,
    )
    density_results.append((density, bin_edges))

global_max_density = max(
    float(density.max())
    for density, _ in density_results
)

fig, axes = plt.subplots(1, 2, figsize=(24, 9), dpi=300)

density_items = [
    (density_results[0], PHI1_LABEL),
    (density_results[1], PHI2_LABEL),
]

for ax, ((density, bin_edges), angle_label) in zip(axes, density_items):
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    ax.plot(
        bin_centers,
        density,
        color="royalblue",
        linewidth=3,
        label=plot_system_name,
    )
    ax.fill_between(
        bin_centers,
        density,
        color="royalblue",
        alpha=0.15,
    )

    ax.set_xlim(*plot_angle_range)
    ax.set_ylim(0, global_max_density * 1.10)
    ax.set_xlabel(f"{angle_label} Angle (degree)", fontsize=30)
    ax.set_ylabel("Probability Density", fontsize=30)
    ax.tick_params(axis="both", which="major", labelsize=24)
    ax.legend(fontsize=18, frameon=False)

fig.tight_layout()

pdf_figure_path = OUTPUT_DIR / "ligand_dihedral_probability_density.png"
fig.savefig(pdf_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved: {pdf_figure_path.resolve()}")


## 19. 完成後確認

- PSF 與 DCD 屬於同一套模擬系統。
- LIGAND_SELECTION 只選到一個 residue。
- PHI1_ATOMS、PHI2_ATOMS 各包含四個 atom names。
- 每個 atom name 在選定 residue 中只出現一次。
- STRIDE = 10 時，取樣間隔為 0.020 ns。
- 若 FOLD_TO_90 = True，角度位於 −90° 到 90°。

若要分析另一種小分子，只需重新執行：

1. 更換 PSF_PATH、DCD_PATH、SYSTEM_NAME。
2. 執行候選小分子掃描。
3. 修改 LIGAND_RESNAME、LIGAND_RESID、PHI1_ATOMS、PHI2_ATOMS。
4. 由檢查小分子 atom names 的 cell 往下執行。
